# DLinear - Walmart Weekly Sales

**DLinear** (Zeng et al., *"Are Transformers Effective for Time Series Forecasting?"*, AAAI 2023)
is a deliberately minimal forecaster: decompose the lookback window into a **trend**
(moving average) and a **seasonal/remainder** component, run a *single linear layer* over
each, and sum them. No attention, no recurrence -- just two `lookback -> horizon` weight
matrices. The paper's point was that this beats heavier Transformer forecasters on long-horizon
benchmarks.

**How this notebook stays comparable to the classical models (ARIMA / SARIMA / Prophet):**

* Metrics are pooled over the same 30 shared **representative series**
  (`representative_series_v1.csv`).
* Same validation protocol: expanding-window CV (`3 x 13` weeks) for model selection, then a
  single `39`-week chronological **holdout** for the winner only.
* Same pooled **WMAE** (holiday weeks weighted 5x) and the same `build_result_row` ->
  `reports/tables/` logging via **MLflow**.

**What is different, on purpose:**

1. DLinear is trained as a **single global, channel-independent** model -- one shared linear map
   applied to every `(Store, Dept)` -- rather than one model per series. It is therefore *fit on
   every clean series in the dataset* (~2.7k of them) and merely *scored* on the 30
   representative ones. This is the honest way to use a global model, and it is the point of
   contrast with the local classical baselines.
2. It uses **target history only**. Markdowns and economic regressors are *not* fed in, so this
   answers: *can a tiny linear net on past sales alone compete?*

> **Why train on all series?** A 52-week lookback predicting 13 weeks needs `52 + 13 = 65` weeks
> just to form **one** training window. The earliest CV fold only has ~65 weeks of history, so
> restricting training to the 30 representative series would yield **30 windows** against ~1,378
> parameters. Pooling all clean series turns that into ~2.7k windows on the same fold.


In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore", category=UserWarning)

# Make the project root importable when running the notebook from notebooks/
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import mlflow

# If your package isn't named "src", change the imports below.
from src.walmart_forecasting.data import load_processed_data
from src.walmart_forecasting.paths import TABLES_DIR
from src.walmart_forecasting.validation import chronological_holdout, expanding_window_splits
from src.walmart_forecasting.experiment import (
    HOLIDAY_WEIGHT,
    NON_HOLIDAY_WEIGHT,
    CV_FOLDS,
    CV_VALIDATION_WEEKS,
    FINAL_HOLDOUT_WEEKS,
    DEFAULT_RANDOM_SEED,
    build_common_parameters,
    build_result_row,
    make_run_name,
)
from src.walmart_forecasting.tracking import mlflow_run

# Reproducibility. Note: neither CPU nor MPS kernels are bit-for-bit deterministic across
# machines, so expect tiny run-to-run drift in the metrics even with the seeds pinned.
np.random.seed(DEFAULT_RANDOM_SEED)
torch.manual_seed(DEFAULT_RANDOM_SEED)

# DLinear is two Linear layers. It is so small that per-kernel launch overhead dominates, and
# on Apple Silicon the CPU actually beats the MPS (Metal) backend here -- benchmarked at
# ~0.24 s/epoch on CPU vs ~0.53 s/epoch on MPS for the largest fold at batch 1024.
# Flip USE_MPS to True to compare for yourself. (This is also why a linear model is the wrong
# reason to reach for a Colab GPU.)
USE_MPS = False
DEVICE = (
    torch.device("mps")
    if USE_MPS and torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Setup complete. torch={torch.__version__}, device={DEVICE}, "
      f"mps_available={torch.backends.mps.is_available()}")

## 0. Experiment configuration

In [ ]:
ARCHITECTURE = "dlinear"
STAGE = "tuning"  # representative_series scope isn't a "final" leaderboard scope
FORECAST_STRATEGY = "global_channel_independent"  # one shared model over all series
EVALUATION_SCOPE = "representative_series"  # metrics pooled over the shared 30
EXPERIMENT_NAME = "DLinear_Training"  # one MLflow experiment per architecture

RESULTS_FILENAME = "dlinear_representative_results.csv"  # not "_final" -- representative scope only
SERIES_PATH = TABLES_DIR / "representative_series_v1.csv"  # shared with ARIMA / SARIMA / Prophet

# DLinear-only training hyperparameters (shared across trials unless a trial overrides them).
EPOCHS = 80
LEARNING_RATE = 1e-2
BATCH_SIZE = 1024  # large batches: the model is tiny, so per-step overhead dominates
MIN_TRAIN_WEEKS = 65  # a series must be long enough to form at least one 52+13 window

## 1. Load processed data, the shared representative series, and the global training pool

Two distinct sets:

* **`rep_df`** -- the 30 shared representative series. *Scoring only.*
* **`pool_df`** -- every series that is safe to train on. *Fitting only.*

A series enters the pool only if its weekly calendar is **gap-free**. Sliding windows assume
contiguous weekly spacing; a series with a missing week would silently produce a window that
splices across a time jump. 605 of the 3,331 series have at least one gap, so they are excluded.

In [ ]:
processed = load_processed_data()
merged_train = processed.train.copy()
merged_train["Date"] = pd.to_datetime(merged_train["Date"])
merged_train = merged_train.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

if not SERIES_PATH.exists():
    raise FileNotFoundError(
        f"{SERIES_PATH} not found -- run the Prophet notebook first (or any notebook using "
        "RepresentativeSeriesSelector.select_and_save), it builds this shared 30-series file."
    )

selected = pd.read_csv(SERIES_PATH)
selected_keys = list(zip(selected["Store"], selected["Dept"]))

print(f"Train rows (all): {len(merged_train):,}")
print(f"Representative (scoring) series: {len(selected_keys)}")
selected[["Store", "Dept", "total_sales", "n_weeks", "volume_tier"]]

In [ ]:
def is_gap_free(dates: pd.Series) -> bool:
    """True if consecutive observations are exactly 7 days apart throughout."""
    deltas = dates.diff().dropna().dt.days
    return bool((deltas == 7).all())


series_calendar = merged_train.groupby(["Store", "Dept"])["Date"]
clean_keys = {
    key for key, dates in series_calendar
    if len(dates) >= MIN_TRAIN_WEEKS and is_gap_free(dates.sort_values())
}

missing_reps = set(selected_keys) - clean_keys
if missing_reps:
    raise RuntimeError(
        f"Representative series excluded from the training pool: {sorted(missing_reps)}"
    )

clean_frame = pd.DataFrame(sorted(clean_keys), columns=["Store", "Dept"])
pool_df = merged_train.merge(clean_frame, on=["Store", "Dept"], how="inner")
pool_df = pool_df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

rep_df = merged_train.merge(
    selected[["Store", "Dept"]], on=["Store", "Dept"], how="inner"
).sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print(f"Training pool: {len(clean_keys):,} gap-free series "
      f"({merged_train.groupby(['Store', 'Dept']).ngroups - len(clean_keys):,} excluded), "
      f"{len(pool_df):,} rows")
print(f"Scoring set:   {len(selected_keys)} representative series, {len(rep_df):,} rows")
print("All 30 representative series are inside the training pool.")

## 2. Weighted MAE (pooled)

In [ ]:
def weighted_mae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weights = np.where(np.asarray(is_holiday, dtype=bool), HOLIDAY_WEIGHT, NON_HOLIDAY_WEIGHT)
    return float(np.average(np.abs(y_true - y_pred), weights=weights))

## 3. The DLinear model

Three tiny `nn.Module`s:

* `MovingAvg` -- a length-preserving moving average (endpoint-padded), the low-pass filter that
  extracts the **trend**. `kernel_size` must be odd so the padding stays symmetric.
* `SeriesDecomp` -- returns `(seasonal, trend)` where `seasonal = x - trend`.
* `DLinear` -- one `Linear(lookback -> horizon)` per component, summed. The weights are shared
  across every series (**channel-independent**), which is what makes this a single global model.

We also include **`NLinear`** (subtract the last value, one linear layer, add it back) as a
cheap normalization-based ablation, since it is ~5 lines and often competitive under
distribution shift.

In [ ]:
class MovingAvg(nn.Module):
    """Length-preserving moving average that isolates the trend component."""

    def __init__(self, kernel_size: int):
        super().__init__()
        if kernel_size % 2 == 0:
            raise ValueError("kernel_size must be odd so endpoint padding stays symmetric.")
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):  # x: [B, L, C]
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, pad, 1)
        x_padded = torch.cat([front, x, end], dim=1)
        return self.avg(x_padded.permute(0, 2, 1)).permute(0, 2, 1)


class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):  # x: [B, L, C]
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend


class DLinear(nn.Module):
    def __init__(self, lookback: int, horizon: int, kernel_size: int = 25):
        super().__init__()
        self.decomp = SeriesDecomp(kernel_size)
        # Channel-independent / weight-shared: a single linear map reused for every series.
        self.linear_seasonal = nn.Linear(lookback, horizon)
        self.linear_trend = nn.Linear(lookback, horizon)

    def forward(self, x):  # x: [B, L, C]
        seasonal, trend = self.decomp(x)
        seasonal = seasonal.permute(0, 2, 1)  # [B, C, L]
        trend = trend.permute(0, 2, 1)
        out = self.linear_seasonal(seasonal) + self.linear_trend(trend)  # [B, C, H]
        return out.permute(0, 2, 1)  # [B, H, C]


class NLinear(nn.Module):
    def __init__(self, lookback: int, horizon: int, kernel_size: int = 25):
        super().__init__()  # kernel_size accepted for a uniform constructor signature
        self.linear = nn.Linear(lookback, horizon)

    def forward(self, x):  # x: [B, L, C]
        last = x[:, -1:, :]
        z = self.linear((x - last).permute(0, 2, 1)).permute(0, 2, 1)
        return z + last


MODELS = {"dlinear": DLinear, "nlinear": NLinear}

## 4. Per-series standardization + sliding windows

Thousands of departments live on wildly different sales scales, so a globally-shared linear map
only makes sense if each series is put on a common scale first. We **z-score every series using
statistics from its own training portion** (never from validation), slide
`(lookback -> horizon)` windows over the standardized history, then invert the transform on the
predictions.

In [ ]:
def series_frames(df):
    """Split a pooled DataFrame into per-(Store, Dept) date-sorted frames."""
    return {key: g.sort_values("Date") for key, g in df.groupby(["Store", "Dept"])}


def fit_series_stats(train_frames):
    """Mean/std per series from its training portion (std floored to avoid divide-by-zero)."""
    stats = {}
    for key, g in train_frames.items():
        values = g["Weekly_Sales"].to_numpy(dtype=float)
        mean = float(values.mean())
        std = float(values.std())
        stats[key] = (mean, std if std > 1e-8 else 1.0)
    return stats


def build_windows(train_frames, stats, lookback, horizon):
    """Sliding (X, Y) windows over each standardized series, pooled across all series."""
    xs, ys = [], []
    for key, g in train_frames.items():
        mean, std = stats[key]
        values = (g["Weekly_Sales"].to_numpy(dtype=float) - mean) / std
        for start in range(0, len(values) - lookback - horizon + 1):
            xs.append(values[start:start + lookback])
            ys.append(values[start + lookback:start + lookback + horizon])
    if not xs:
        return np.empty((0, lookback, 1), dtype=np.float32), np.empty((0, horizon, 1), dtype=np.float32)
    x = np.asarray(xs, dtype=np.float32)[:, :, None]  # [N, L, 1]
    y = np.asarray(ys, dtype=np.float32)[:, :, None]  # [N, H, 1]
    return x, y

## 5. Train / predict helpers

The loss is **L1 (MAE)** rather than MSE -- it lines up with the WMAE we actually report and is
less swayed by the sharp holiday spikes. Prediction takes the last `lookback` weeks of a series'
training history and emits the whole `horizon` block in one shot (direct multi-step, the way
DLinear is meant to be used), then inverts the per-series standardization.

In [ ]:
def train_model(model, x, y, epochs, lr, batch_size, device):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()
    loader = DataLoader(
        TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
        batch_size=batch_size, shuffle=True,
    )
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def predict_block(model, train_frames, stats, lookback, device):
    """One horizon-length forecast per series, from that series' most recent lookback weeks."""
    model.eval()
    preds = {}
    for key, g in train_frames.items():
        mean, std = stats[key]
        values = (g["Weekly_Sales"].to_numpy(dtype=float) - mean) / std
        if len(values) < lookback:
            continue  # not enough history to form an input window
        window = values[-lookback:][None, :, None].astype(np.float32)
        out = model(torch.from_numpy(window).to(device)).cpu().numpy().reshape(-1)  # [H]
        preds[key] = out * std + mean  # invert standardization
    return preds

## 6. Shared date-partitioned splits (global model)

Because DLinear is one global model, the folds are defined **once** on the shared weekly
calendar -- via the same `chronological_holdout` / `expanding_window_splits` utilities the other
notebooks use -- and every series is then sliced by those same date boundaries. `holdout_split`
is the final 39-week block we touch exactly once at the end.

In [ ]:
holdout_split = chronological_holdout(pool_df, validation_weeks=FINAL_HOLDOUT_WEEKS)
cv_splits = expanding_window_splits(
    holdout_split.train, n_splits=CV_FOLDS, validation_weeks=CV_VALIDATION_WEEKS
)

print(f"Holdout: {holdout_split.validation_start.date()} -> {holdout_split.validation_end.date()} "
      f"({holdout_split.validation_weeks} weeks)")
for i, fold in enumerate(cv_splits):
    weeks = int(fold.train.groupby(["Store", "Dept"]).size().median())
    print(f"  CV fold {i}: ~{weeks} train weeks/series, "
          f"val {fold.validation_start.date()} -> {fold.validation_end.date()}")

## 7. Shared feature-set / preprocessing IDs

In [ ]:
BASE_FEATURE_SET = "target_only_v1"  # DLinear consumes past sales only, no exogenous inputs
BASE_PREPROCESSING = "per_series_zscore_windows_v1"

## 8. Trial runner

One trial = one hyperparameter choice (model type / lookback / trend kernel). For each CV fold
we z-score on that fold's training portion, build windows across the **whole pool**, train a
**fresh** global model, then forecast the 13-week block for each of the **30 representative
series** and pool those into a single WMAE. The holdout path is identical but with `horizon=39`,
and runs only for the winner.

In [ ]:
def _pool_predictions(pred_by_series, val_frames):
    rows = []
    for key, val in val_frames.items():
        if key not in pred_by_series:
            continue
        actual = val["Weekly_Sales"].to_numpy(dtype=float)
        rows.append(pd.DataFrame({
            "Store": key[0], "Dept": key[1], "Date": val["Date"].to_numpy(),
            "actual": actual, "prediction": pred_by_series[key][:len(actual)],
            "IsHoliday": val["IsHoliday"].to_numpy(),
        }))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def _fit_predict_split(model_name, split, lookback, horizon, kernel_size, eval_keys):
    """Fit on every pooled series in split.train; score only eval_keys on split.validation."""
    train_frames = series_frames(split.train)
    stats = fit_series_stats(train_frames)
    x, y = build_windows(train_frames, stats, lookback, horizon)
    if len(x) == 0:
        raise RuntimeError(f"No training windows for lookback={lookback}, horizon={horizon}.")

    torch.manual_seed(DEFAULT_RANDOM_SEED)
    model = MODELS[model_name](lookback=lookback, horizon=horizon, kernel_size=kernel_size)

    fit_start = time.perf_counter()
    train_model(model, x, y, EPOCHS, LEARNING_RATE, BATCH_SIZE, DEVICE)
    fit_seconds = time.perf_counter() - fit_start

    eval_frames = {k: train_frames[k] for k in eval_keys if k in train_frames}
    predict_start = time.perf_counter()
    pred_by_series = predict_block(model, eval_frames, stats, lookback, DEVICE)
    predict_seconds = time.perf_counter() - predict_start

    val_frames = {
        k: g for k, g in series_frames(split.validation).items() if k in set(eval_keys)
    }
    pooled = _pool_predictions(pred_by_series, val_frames)
    return pooled, fit_seconds, predict_seconds, len(x)


def run_dlinear_trial(label, model_name, lookback, kernel_size,
                      feature_set, preprocessing, do_cv=True, do_holdout=False):
    eval_keys = selected_keys
    total_fit_seconds = 0.0
    total_predict_seconds = 0.0
    result = {
        "label": label, "model_name": model_name, "lookback": lookback,
        "kernel_size": kernel_size, "feature_set": feature_set, "preprocessing": preprocessing,
        "series_count": len(eval_keys),
    }

    if do_cv:
        fold_wmaes, fold_windows = [], []
        for fold in cv_splits:
            pooled, fit_s, pred_s, n_windows = _fit_predict_split(
                model_name, fold, lookback, CV_VALIDATION_WEEKS, kernel_size, eval_keys
            )
            total_fit_seconds += fit_s
            total_predict_seconds += pred_s
            fold_windows.append(n_windows)
            fold_wmaes.append(
                weighted_mae(pooled["actual"], pooled["prediction"], pooled["IsHoliday"])
            )
        result["cv_wmae_mean"] = float(np.mean(fold_wmaes))
        result["cv_wmae_std"] = float(np.std(fold_wmaes))
        result["fold_wmaes"] = fold_wmaes
        result["fold_windows"] = fold_windows

    if do_holdout:
        pooled, fit_s, pred_s, n_windows = _fit_predict_split(
            model_name, holdout_split, lookback, FINAL_HOLDOUT_WEEKS, kernel_size, eval_keys
        )
        total_fit_seconds += fit_s
        total_predict_seconds += pred_s
        errors = pooled["actual"] - pooled["prediction"]
        result["holdout_wmae"] = weighted_mae(pooled["actual"], pooled["prediction"], pooled["IsHoliday"])
        result["holdout_mae"] = float(np.mean(np.abs(errors)))
        result["holdout_rmse"] = float(np.sqrt(np.mean(errors ** 2)))
        result["holdout_windows"] = n_windows
        result["pooled_holdout_predictions"] = pooled

    result["fit_seconds"] = total_fit_seconds
    result["predict_seconds"] = total_predict_seconds

    summary = []
    if do_cv:
        summary.append(f"cv_wmae_mean={result['cv_wmae_mean']:,.1f}")
        summary.append(f"windows/fold={result['fold_windows']}")
    if do_holdout:
        summary.append(f"holdout_wmae={result['holdout_wmae']:,.1f}")
    print(f"[{label}] " + ", ".join(summary) + f", fit={total_fit_seconds:.1f}s")
    return result

## 9. Timing calibration -- run this before committing to the full loop

One fold of the base config, to sanity-check runtime and confirm the pipeline produces a finite
WMAE before launching all four trials.

In [ ]:
calib_start = time.perf_counter()
_pooled, _fit_s, _pred_s, _n_windows = _fit_predict_split(
    "dlinear", cv_splits[0], lookback=52, horizon=CV_VALIDATION_WEEKS,
    kernel_size=25, eval_keys=selected_keys,
)
_calib_wmae = weighted_mae(_pooled["actual"], _pooled["prediction"], _pooled["IsHoliday"])
print(f"1 fold: {time.perf_counter() - calib_start:.1f}s wall, fit={_fit_s:.1f}s, "
      f"{_n_windows:,} training windows, fold WMAE={_calib_wmae:,.1f}")
print(f"Rough estimate for 4 trials x {CV_FOLDS} folds: ~{_fit_s * 4 * CV_FOLDS / 60:.1f} min")

## 10. Trials (CV only) -- model type, lookback, and trend-kernel ablations

* **A** DLinear, 52-week lookback (a full annual cycle), kernel 25.
* **B** DLinear, 26-week lookback -- shorter memory, but many more training windows.
* **C** NLinear, 52-week lookback -- normalization-based ablation.
* **D** DLinear, 52-week lookback, kernel 51 -- a much smoother trend decomposition.

`windows/fold` is printed for each trial: it is the honest measure of how much data each
configuration actually got to learn from.

In [ ]:
trial_a = run_dlinear_trial(
    "dlinear_L52_k25", model_name="dlinear", lookback=52, kernel_size=25,
    feature_set=BASE_FEATURE_SET, preprocessing=BASE_PREPROCESSING,
)
trial_b = run_dlinear_trial(
    "dlinear_L26_k13", model_name="dlinear", lookback=26, kernel_size=13,
    feature_set=BASE_FEATURE_SET, preprocessing=BASE_PREPROCESSING,
)
trial_c = run_dlinear_trial(
    "nlinear_L52", model_name="nlinear", lookback=52, kernel_size=25,
    feature_set=BASE_FEATURE_SET, preprocessing=BASE_PREPROCESSING,
)
trial_d = run_dlinear_trial(
    "dlinear_L52_k51", model_name="dlinear", lookback=52, kernel_size=51,
    feature_set=BASE_FEATURE_SET, preprocessing=BASE_PREPROCESSING,
)

## 11. Compare all trials, pick the winner by CV WMAE

In [ ]:
trials = [trial_a, trial_b, trial_c, trial_d]

comparison = pd.DataFrame([
    {"trial": t["label"], "model": t["model_name"], "lookback": t["lookback"],
     "kernel_size": t["kernel_size"], "cv_wmae_mean": t["cv_wmae_mean"],
     "cv_wmae_std": t["cv_wmae_std"], "windows_fold0": t["fold_windows"][0]}
    for t in trials
])
comparison.sort_values("cv_wmae_mean")

In [ ]:
winner = min(trials, key=lambda t: t["cv_wmae_mean"])
print(f"Winner by CV WMAE: {winner['label']} (cv_wmae_mean={winner['cv_wmae_mean']:,.1f})")

FEATURE_SET = winner["feature_set"]
PREPROCESSING = winner["preprocessing"]
TRIAL_NAME = winner["label"].lower()

## 12. Final holdout -- evaluated exactly once, only for the winner

Retrain the winning config on the full pre-holdout window and forecast the untouched 39 weeks.

In [ ]:
run_name = make_run_name(
    architecture=ARCHITECTURE,
    stage=STAGE,
    feature_set=FEATURE_SET,
    trial_name=TRIAL_NAME,
)

common_params = build_common_parameters(
    architecture=ARCHITECTURE,
    stage=STAGE,
    feature_set=FEATURE_SET,
    preprocessing=PREPROCESSING,
    evaluation_scope=EVALUATION_SCOPE,
    forecast_strategy=FORECAST_STRATEGY,
    series_count=len(selected_keys),
    random_seed=DEFAULT_RANDOM_SEED,
    extra_parameters={
        "model_name": winner["model_name"],
        "lookback": winner["lookback"],
        "kernel_size": winner["kernel_size"],
        "horizon_cv": CV_VALIDATION_WEEKS,
        "horizon_holdout": FINAL_HOLDOUT_WEEKS,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "loss": "l1",
        "device": str(DEVICE),
        "training_pool_series": len(clean_keys),
        "trials_compared": [t["label"] for t in trials],
    },
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
results_path = TABLES_DIR / RESULTS_FILENAME

with mlflow_run(
    experiment_name=EXPERIMENT_NAME,
    run_name=run_name,
    parameters=common_params,
    tags={"architecture": ARCHITECTURE, "stage": STAGE, "scope": EVALUATION_SCOPE, "trial": TRIAL_NAME},
) as run:

    holdout_only = run_dlinear_trial(
        winner["label"] + "_final_holdout", model_name=winner["model_name"],
        lookback=winner["lookback"], kernel_size=winner["kernel_size"],
        feature_set=FEATURE_SET, preprocessing=PREPROCESSING, do_cv=False, do_holdout=True,
    )

    final_metrics = {
        "cv_wmae_mean": winner["cv_wmae_mean"],
        "cv_wmae_std": winner["cv_wmae_std"],
        "holdout_wmae": holdout_only["holdout_wmae"],
        "holdout_mae": holdout_only["holdout_mae"],
        "holdout_rmse": holdout_only["holdout_rmse"],
    }

    total_fit_seconds = winner["fit_seconds"] + holdout_only["fit_seconds"]
    total_predict_seconds = winner["predict_seconds"] + holdout_only["predict_seconds"]

    mlflow.log_metrics({**final_metrics, "fit_seconds": total_fit_seconds, "predict_seconds": total_predict_seconds})

    holdout_only["pooled_holdout_predictions"].to_csv(TABLES_DIR / "dlinear_holdout_predictions.csv", index=False)
    mlflow.log_artifact(str(TABLES_DIR / "dlinear_holdout_predictions.csv"))

print("\nFinal metrics (winner, evaluated once on holdout):")
for key, value in final_metrics.items():
    print(f"  {key}: {value:,.2f}")

## 13. Save to the shared representative-results CSV

In [ ]:
result_row = build_result_row(
    architecture=ARCHITECTURE,
    run_name=run_name,
    stage=STAGE,
    tracker="mlflow",
    feature_set=FEATURE_SET,
    preprocessing=PREPROCESSING,
    evaluation_scope=EVALUATION_SCOPE,
    forecast_strategy=FORECAST_STRATEGY,
    series_count=len(selected_keys),
    metrics=final_metrics,
    fit_seconds=total_fit_seconds,
    predict_seconds=total_predict_seconds,
    notes=(
        f"DLinear, winning config from 4 trials (lookback 52/26, NLinear, trend-kernel 25/51 "
        f"ablations): {winner['label']}. Single global channel-independent linear model fit on "
        f"{len(clean_keys)} gap-free series, scored on the 30 representative ones. Target history "
        f"only (no exogenous inputs), per-series z-score windows, L1 loss. Holdout evaluated once."
    ),
)

results_row_df = pd.DataFrame([result_row])
if results_path.exists():
    existing = pd.read_csv(results_path)
    results_row_df = pd.concat([existing, results_row_df], ignore_index=True)
results_row_df.to_csv(results_path, index=False)
print(f"Saved winning result to {results_path.name}")


for trial in trials:
    if trial["label"] == winner["label"]:
        continue  # already logged above with the full run

    trial_run_name = make_run_name(
        architecture=ARCHITECTURE, stage=STAGE, feature_set=trial["feature_set"], trial_name=trial["label"].lower()
    )
    trial_params = build_common_parameters(
        architecture=ARCHITECTURE, stage=STAGE, feature_set=trial["feature_set"], preprocessing=trial["preprocessing"],
        evaluation_scope=EVALUATION_SCOPE, forecast_strategy=FORECAST_STRATEGY, series_count=len(selected_keys),
        extra_parameters={"model_name": trial["model_name"], "lookback": trial["lookback"], "kernel_size": trial["kernel_size"]},
    )
    with mlflow_run(
        experiment_name=EXPERIMENT_NAME, run_name=trial_run_name, parameters=trial_params,
        tags={"architecture": ARCHITECTURE, "stage": STAGE, "trial": trial["label"].lower()},
    ):
        mlflow.log_metrics({
            "cv_wmae_mean": trial["cv_wmae_mean"], "cv_wmae_std": trial["cv_wmae_std"],
            "fit_seconds": trial["fit_seconds"], "predict_seconds": trial["predict_seconds"],
        })

print("Logged non-winning trials to MLflow (CV-only, no holdout).")

## 14. A few forecasts, visually

In [ ]:
sample_series = holdout_only["pooled_holdout_predictions"].groupby(["Store", "Dept"])
keys_to_plot = list(sample_series.groups.keys())[:6]

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
for ax, key in zip(axes.ravel(), keys_to_plot):
    g = sample_series.get_group(key).sort_values("Date")
    ax.plot(g["Date"], g["actual"], label="actual", marker="o", ms=3)
    ax.plot(g["Date"], g["prediction"], label="DLinear", marker="x", ms=3)
    holiday = g[g["IsHoliday"]]
    ax.scatter(holiday["Date"], holiday["actual"], color="red", zorder=5, s=30, label="holiday")
    ax.set_title(f"Store {key[0]} Dept {key[1]}")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=8)

fig.suptitle("DLinear 39-week holdout forecasts (winner)", fontsize=13)
fig.tight_layout()
plt.show()

## 15. Takeaways & next steps

* **DLinear vs the classical models.** Compare `holdout_wmae` here against
  `arima_representative_results.csv` / `sarima_*` / `prophet_*` -- all scored on the *same* 30
  series. Because DLinear sees **target history only**, this measures how far a tiny global
  linear net gets without the economic/markdown regressors that ARIMAX and the GBMs lean on.
  It is also a *global* model fit on ~2.7k series, whereas the classical baselines are fit
  per-series; call that out explicitly rather than reporting the WMAE gap alone.
* **Window scarcity, not model capacity, is the binding constraint.** With only ~65 training
  weeks in the earliest CV fold, a 52-week lookback forms a single window *per series* -- which
  is precisely why the model is pooled over the whole dataset instead of the 30 series alone.
  The `windows_fold0` column in the comparison table makes this visible.
* **No GPU needed -- and MPS is actually slower here.** Benchmarked on the largest fold at batch
  1024: ~0.24 s/epoch on CPU vs ~0.53 s/epoch on MPS, because kernel-launch overhead dwarfs the
  arithmetic for a model this small. Reaching for a Colab GPU would be strictly worse. Save that
  for the heavier nets (LSTM/TCN/PatchTST).
* **Natural extensions:** add RevIN normalization, feed exogenous regressors (moving toward a
  covariate-aware linear model), or score the same global model on *all* 3,331 series for a
  full-dataset (`_final`) leaderboard entry.
